# Bonus B6 solution, notebook 1: two playlists on one index

The worked answers to the challenge of `b6a_meet_pinecone.ipynb`. The setup repeats what the tasks need from the
notebook (the two clients and `embed`), so this file runs on its own. It needs the same two keys.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The keys come from the environment or .env, and are never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."
os.environ.setdefault("PINECONE_API_KEY", local.get("PINECONE_API_KEY") or "")
assert os.environ["PINECONE_API_KEY"], "Put PINECONE_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")
print("Pinecone key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes
Pinecone key found: yes


In [2]:
import time
import numpy as np
from openai import OpenAI
from pinecone import Pinecone, DenseVectorQuery

client = OpenAI(timeout=60)
pc = Pinecone()
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]
print("embedding model:", EMBEDDING_MODEL)


def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding

embedding model: text-embedding-3-small


The songs, one dictionary per friend:

In [3]:
PLAYLISTS = {
    "layla": {
        "night_bus": "A slow piano ballad about missing someone who lives far away.",
        "paper_boats": "An upbeat pop song about dancing at a summer wedding.",
        "static": "A loud rock song about quitting a boring job.",
    },
    "omar": {
        "long_road": "A quiet acoustic guitar song about heartbreak on the motorway at night.",
        "sunrise_run": "A fast electronic track for a morning workout.",
        "old_radio": "A cheerful folk tune about a grandfather's kitchen.",
    },
}

---

## Task 1: Build the index

The same `create` as in Part B of the notebook, with the new name. The vectors of the embedding model have 1,536
numbers.

In [4]:
INDEX_NAME = "ai-agents-b6-playlists"

if not pc.indexes.exists(INDEX_NAME):
    pc.indexes.create(
        name=INDEX_NAME,
        schema={"fields": {"embedding": {"type": "dense_vector", "dimension": 1536, "metric": "cosine"}}},
        deployment={"deployment_type": "managed", "cloud": "aws", "region": "us-east-1"},
    )

index = pc.index(INDEX_NAME)
print("exists:", pc.indexes.exists(INDEX_NAME))

exists: True


One upsert per friend. The namespace is the friend's name, and writing to it creates it. Then we wait until both
namespaces hold three records, with `stored` from Part C.

In [5]:
for friend, songs in PLAYLISTS.items():
    documents = [{"_id": name, "embedding": embed(text), "text": text} for name, text in songs.items()]
    written = index.documents.upsert(namespace=friend, documents=documents)
    print(friend, "upserted:", written.upserted_count)

def stored(namespace):
    spaces = index.describe_index_stats().namespaces
    return spaces[namespace].vector_count if namespace in spaces else 0


while stored("layla") < 3 or stored("omar") < 3:
    time.sleep(1)

print("layla:", stored("layla"), "| omar:", stored("omar"))

layla upserted: 3


omar upserted: 3


layla: 3 | omar: 3


Three records in each namespace, six in the index. The two playlists share an index and nothing else.

---

## Task 2: Search each friend's playlist

`search` here takes the namespace as its first argument, because that is what changes between the calls.

In [6]:
def search(namespace, question, k=1):
    got = index.documents.search(
        namespace=namespace,
        top_k=k,
        score_by=[DenseVectorQuery(field="embedding", values=embed(question))],
        include_fields=["text"],
    )
    return got.matches


SAD = "A sad song for a long drive"

for friend in PLAYLISTS:
    for match in search(friend, SAD):
        print(f"{friend:6} {match.score:.3f}  {match.id:12} {match.text}")

layla  0.548  night_bus    A slow piano ballad about missing someone who lives far away.


omar   0.644  long_road    A quiet acoustic guitar song about heartbreak on the motorway at night.


`layla` returns `night_bus` at 0.548, and `omar` returns `long_road` at 0.644. The same question got a different song from each namespace: each search read one namespace and ranked only the three songs in it. Omar's guitar song scores higher than Layla's ballad, and the search in `layla` did not rank it.

---

## Task 3: Search the wrong playlist

The fast track for a workout is on Omar's playlist. We ask Layla's.

In [7]:
GYM = "A fast track for the gym"

for match in search("layla", GYM, k=3):
    print(f"layla  {match.score:.3f}  {match.id:12} {match.text}")

for match in search("omar", GYM):
    print(f"omar   {match.score:.3f}  {match.id:12} {match.text}")

layla  0.235  static       A loud rock song about quitting a boring job.
layla  0.193  paper_boats  An upbeat pop song about dancing at a summer wedding.
layla  0.118  night_bus    A slow piano ballad about missing someone who lives far away.


omar   0.632  sunrise_run  A fast electronic track for a morning workout.


Layla's playlist returns `static` first, at 0.235: a loud rock song about quitting a job. Pinecone raised no error and gave no warning. It ranked the three songs it was asked to rank, and the best of them is a poor match. The only sign is the score, far under the 0.548 and 0.644 of Task 2. The track we were looking for is `sunrise_run`, in `omar`, at 0.632.

**A search in the wrong namespace looks like a search that worked.** When a result surprises you, check the namespace of the write and of the search first. (A namespace that does not exist returns an empty list of matches, also with no error.)

---

## Task 4: Delete the index

In [8]:
pc.indexes.delete(INDEX_NAME)

print("exists:", pc.indexes.exists(INDEX_NAME))

exists: False


`exists: False`. Both namespaces went with the index.